In [1]:
from nemo.collections.asr.parts.submodules.wfst_decoder import RivaGpuWfstDecoder
import riva.asrlib.decoder.python_decoder as riva_decoder
from inference_funcs import load_bit_phoneme_model, evaluate_model
from dataset import getDatasetLoaders
import numpy as np
import torch
import torch.nn.functional as F
import pickle

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_word_symbols(path):
    word_id_to_word_str = {}
    with open(path, "rt") as fh:
        for line in fh:
            word_str, word_id = line.rstrip().split()
            word_id_to_word_str[int(word_id)] = word_str
    return word_id_to_word_str

In [3]:
bit_phoneme_filepath = "/data/models/transformer_short_training_fixed_seed_0/"
device = 'cuda'
with open('/data/text/validation_sentences_ground_truth.pkl', 'rb') as f:
    val_ground_truth_all = pickle.load(f)
log_probs_all = torch.load(f"{bit_phoneme_filepath}log_probs_arranged.pth").to(dtype=torch.float32, device=device)
lengths_all = torch.load(f"{bit_phoneme_filepath}log_probs_length.pth").to(dtype=torch.int64, device='cpu')

In [4]:
log_probs = log_probs_all[399:401]
lengths = lengths_all[399:401]
val_ground_truth = val_ground_truth_all[399:401]

In [5]:
lm_fst_path = "/data/code/nejm-brain-to-text/language_model/pretrained_language_models/openwebtext_1gram_lm_sil/TLG.fst"
symbol_table_path = '/data/lm/words.txt'
num_tokens = 41
#config = riva_decoder.BatchedMappedOnlineDecoderCudaConfig()
#print(dir(config))
#print(config.use_lattice_postprocessor)
#config.lattice_postprocessor_opts.nbest = 2

In [9]:
from riva.asrlib.decoder.python_decoder import BatchedMappedDecoderCuda, BatchedMappedOnlineDecoderCuda, BatchedMappedDecoderCudaConfig
import multiprocessing
def create_decoder_config():
    config = BatchedMappedDecoderCudaConfig()
    config.n_input_per_chunk = 50
    config.online_opts.decoder_opts.default_beam = 18.0
    config.online_opts.decoder_opts.lattice_beam = 8.0
    config.online_opts.decoder_opts.max_active = 10_000
    config.online_opts.decoder_opts.ntokens_pre_allocated = 10_000_000
    config.online_opts.determinize_lattice = True
    config.online_opts.max_batch_size = 200
    config.online_opts.num_channels = config.online_opts.max_batch_size * 2
    config.online_opts.frame_shift_seconds = 0.04
    config.online_opts.lattice_postprocessor_opts.acoustic_scale = 0.8
    config.online_opts.lattice_postprocessor_opts.lm_scale = 1.0
    config.online_opts.lattice_postprocessor_opts.word_ins_penalty = 0.0
    config.online_opts.lattice_postprocessor_opts.nbest = 1
    config.online_opts.decoder_opts.blank_penalty = 0.7
    config.online_opts.num_decoder_copy_threads = 2
    config.online_opts.num_post_processing_worker_threads = (
        multiprocessing.cpu_count() - config.online_opts.num_decoder_copy_threads
    )

    return config

offline_config = create_decoder_config()
offline_config.online_opts.max_batch_size = 16
config = offline_config.online_opts

In [10]:
decoder = riva_decoder.BatchedMappedOnlineDecoderCuda(
    config, lm_fst_path, symbol_table_path, num_tokens
)

In [90]:
batch_size = log_probs.shape[0]
corr_ids = list(range(batch_size))

for corr_id in corr_ids:
    success = decoder.try_init_corr_id(corr_id)
    # Do SetLatticeCallback() here if you want
    # Is there some way that I can get the lattice other than callbacks?
    assert success
    
log_probs_list = [0] * batch_size
is_first_chunk = [0] * batch_size
is_last_chunk = [0] * batch_size
    
for i in range(batch_size):
    log_probs_list[i] = log_probs[i, :lengths[i], :]
    is_first_chunk[i] = True
    is_last_chunk[i]  = True
    
channels, full_partial_hypotheses = \
                    decoder.decode_batch(corr_ids, log_probs_list,
                                         is_first_chunk, is_last_chunk)

for corr_id in corr_ids:
    success = decoder.try_init_corr_id(corr_id)
    # Do SetLatticeCallback() here if you want
    # Is there some way that I can get the lattice other than callbacks?
    assert success
    
for i in range(batch_size):
    log_probs_list[i] = log_probs[i, :lengths[i] // 2, :]
    is_first_chunk[i] = True
    is_last_chunk[i]  = False
    
channels, chunked_partial_hypotheses1 = \
    decoder.decode_batch(corr_ids, log_probs_list,
                            is_first_chunk, is_last_chunk)
    

for i in range(batch_size):
    log_probs_list[i] = log_probs[i, lengths[i] // 2:, :]
    is_first_chunk[i] = False
    is_last_chunk[i]  = True
channels, chunked_partial_hypotheses2 = \
    decoder.decode_batch(corr_ids, log_probs_list,
                            is_first_chunk, is_last_chunk)


In [91]:
print(dir(chunked_partial_hypotheses2[0]))

['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'ilabels', 'score', 'word_end_times_frames', 'word_start_times_frames', 'words']


In [92]:
print(full_partial_hypotheses.__le__)

<method-wrapper '__le__' of list object at 0x7b7e1457e500>


In [93]:
id2word = load_word_symbols(symbol_table_path)

In [94]:
idx = 1
print(val_ground_truth[idx])
print([id2word[i] for i in full_partial_hypotheses[idx].words])
print(full_partial_hypotheses[idx].score)


friday afternoon at five thirty
['VOID', 'AFFERENT', 'AT', 'FIVE', 'LADY']
-inf


In [52]:
logits_list = [log_probs_arranged[400, 0:4]]
corr_ids = [0]
is_first_chunk = [True]
is_last_chunk = [False]

In [53]:
channels, partial_hypotheses = streaming_decoder.decode_batch(
    corr_ids=corr_ids,
    logits_t=logits_list,
    is_first_chunk=is_first_chunk,
    is_last_chunk=is_last_chunk
)

TypeError: decode_batch(): incompatible function arguments. The following argument types are supported:
    1. decode_batch(self, arg0: list[int], arg1: list[ndarray[dtype=float32, shape=(*, *), order='C', device='cuda']], arg2: list[bool], arg3: list[bool], /) -> tuple[list[int], list[riva.asrlib.decoder.python_decoder.PartialHypothesisEx]]

Invoked with types: riva.asrlib.decoder.python_decoder.BatchedMappedOnlineDecoderCuda, kwargs = { corr_ids: list, logits_t: list, is_first_chunk: list, is_last_chunk: list }

In [10]:
hypotheses = torch.load('/data/models/transformer_short_training_fixed_seed_0/hypotheses.pth')

In [11]:
print(dir(hypotheses[0]))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_has_alignment', '_has_timesteps', '_hypotheses', '_shape0', '_shape1', 'has_alignment', 'has_timesteps', 'replace_unit_', 'shape0', 'shape1']


In [19]:
idx = 700
print(val_ground_truth[idx])
for i in range(18):
    print(hypotheses[idx]._hypotheses[i])

i'm kind of a car buff myself
WfstNbestUnit(words=("I'M", 'KIND', 'OF', 'A', 'CAR', 'BY', 'MYSELF'), timesteps=(0, 9, 15, 26, 35, 39, 52), alignment=(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 23, 23, 1, 1, 21, 7, 24, 24, 10, 10, 1, 1, 4, 36, 36, 1, 1, 1, 0, 0, 4, 4, 0, 1, 1, 21, 21, 2, 29, 29, 1, 1, 8, 8, 7, 0, 0, 0, 1, 1, 23, 23, 23, 7, 30, 30, 30, 12, 22, 22, 15, 15, 1, 1), score=97606.734375)
WfstNbestUnit(words=("I'M", 'KIND', 'OF', 'A', 'CAR', 'BUT', 'MYSELF'), timesteps=(0, 9, 15, 26, 35, 39, 48), alignment=(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 23, 23, 1, 1, 21, 7, 24, 24, 10, 10, 1, 1, 4, 36, 36, 1, 1, 1, 0, 0, 4, 4, 0, 1, 1, 21, 21, 2, 29, 29, 1, 1, 8, 8, 4, 0, 32, 0, 1, 1, 23, 23, 23, 7, 30, 30, 30, 12, 22, 22, 15, 15, 1, 1), score=98687.4296875)
WfstNbestUnit(words=("I'M", 'KIND', 'OF', 'A', 'CAR', 'BOTH', 'MYSELF'), timesteps=(0, 9, 15, 26, 35, 39, 52), alignment=(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 23, 23, 1, 1, 21, 7, 24, 24, 10, 10, 1, 1, 4, 36, 36, 1, 1, 1, 0, 0, 4, 4, 0, 1, 1,

In [36]:
# First second only (10 frames = 1s if each frame = 100ms)

log_probs_chunk = log_probs_arranged[:, 0:10, :]  # take first 10 frames
log_probs_len_chunk = torch.tensor([10] * log_probs_arranged.shape[0])  # length per batch

# Decode just this first-second chunk
hypotheses_first_second = decoder._decode_nbest(log_probs_chunk, log_probs_len_chunk)


log_probs_chunk = log_probs_arranged[:, 10:20, :]  # take first 10 frames
log_probs_len_chunk = torch.tensor([10] * log_probs_arranged.shape[0])  # length per batch

hypotheses_second_second = decoder._decode_nbest(log_probs_chunk, log_probs_len_chunk)